# SPARQL Query Runner for Fuseki Server

Executes SPARQL queries against a Fuseki server using HTTP requests.

### Parameters:
- **fuseki_server**: Fuseki server URL
- **graph_name**: Target dataset name
- **query_names**: List of query files to execute

In [ ]:
from helpers.fuseki_config import query_endpoint, update_endpoint
from helpers.fuseki_queries import execute_sparql_query, execute_sparql_update, format_results, load_query

In [ ]:
# Configuration
query_names = [
    # "position-secuences.sparql",
    "temporal-context-annotation.sparql",
    "meaningful-interaction.sparql",
    "origin-destination-interaction.sparql",
    "passing-interaction.sparql"
]

In [ ]:
import re
from collections import defaultdict


def link_position_sequences(query_endpoint: str, update_endpoint: str, batch_size: int = 500) -> dict:
    """
    Create hasNextPosition/hasPreviousPosition links between consecutive positions
    and add hasPositionIndex to each position.

    Returns statistics dict with counts.
    """

    # Step 1: Query all positions with their trajectories
    print("Step 1: Fetching all positions...")
    query = """
    PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
    PREFIX datacron: <http://www.datacron-project.eu/datAcron#>

    SELECT ?pos ?trajectory
    WHERE {
        ?pos dul:isPartOf ?trajectory .
        ?trajectory a datacron:Trajectory .
    }
    """

    results = execute_sparql_query(query_endpoint, query)
    df = format_results(results)

    if df.empty:
        print("No positions found.")
        return {"positions": 0, "links_created": 0}

    print(f"   Found {len(df)} positions")

    # Step 2: Parse index from position URI and group by trajectory
    print("Step 2: Parsing indices and grouping by trajectory...")

    def extract_index(pos_uri: str) -> int:
        """Extract sequence index from position URI (e.g., rawPosition_123_45 -> 45)"""
        match = re.search(r'rawPosition_\d+_(\d+)$', pos_uri)
        if match:
            return int(match.group(1))
        # Fallback: try to get last number after underscore
        match = re.search(r'_(\d+)$', pos_uri)
        return int(match.group(1)) if match else -1

    # Group positions by trajectory
    trajectory_positions = defaultdict(list)
    for _, row in df.iterrows():
        pos_uri = row['pos']
        trajectory_uri = row['trajectory']
        idx = extract_index(pos_uri)
        if idx >= 0:
            trajectory_positions[trajectory_uri].append((idx, pos_uri))

    print(f"   Grouped into {len(trajectory_positions)} trajectories")

    # Step 3: Sort and create consecutive pairs + collect all positions with indices
    print("Step 3: Creating consecutive position pairs and collecting indices...")

    pairs = []  # [(pos1_uri, pos2_uri), ...]
    position_indices = []  # [(pos_uri, index), ...]

    for _traj_uri, positions in trajectory_positions.items():
        # Sort by index
        positions.sort(key=lambda x: x[0])

        # Collect all positions with their indices
        for idx, pos_uri in positions:
            position_indices.append((pos_uri, idx))

        # Create consecutive pairs
        for i in range(len(positions) - 1):
            pos1_uri = positions[i][1]
            pos2_uri = positions[i + 1][1]
            pairs.append((pos1_uri, pos2_uri))

    print(f"   Found {len(pairs)} consecutive pairs")
    print(f"   Found {len(position_indices)} positions to index")

    if not pairs and not position_indices:
        return {"positions": len(df), "trajectories": len(trajectory_positions), "links_created": 0, "indices_added": 0}

    # Step 4: Batch insert the position indices
    print(f"Step 4: Inserting position indices in batches of {batch_size}...")

    total_indices_inserted = 0
    for batch_start in range(0, len(position_indices), batch_size):
        batch = position_indices[batch_start:batch_start + batch_size]

        # Build INSERT DATA query for indices
        insert_statements = []
        for pos_uri, idx in batch:
            insert_statements.append(f"  <{pos_uri}> trajecto:hasPositionIndex {idx} .")

        insert_query = f"""PREFIX trajecto: <http://w3id.org/trajecto#>

INSERT DATA {{
{chr(10).join(insert_statements)}
}}"""

        success = execute_sparql_update(update_endpoint, insert_query)
        if success:
            total_indices_inserted += len(batch)
            print(f"   Index batch {batch_start // batch_size + 1}: inserted {len(batch)} triples")
        else:
            print(f"   Index batch {batch_start // batch_size + 1}: FAILED")

    # Step 5: Batch insert the sequence links
    print(f"Step 5: Inserting sequence links in batches of {batch_size}...")

    total_links_inserted = 0
    for batch_start in range(0, len(pairs), batch_size):
        batch = pairs[batch_start:batch_start + batch_size]

        # Build INSERT DATA query for links
        insert_statements = []
        for pos1, pos2 in batch:
            insert_statements.append(f"  <{pos1}> trajecto:hasNextPosition <{pos2}> .")
            insert_statements.append(f"  <{pos2}> trajecto:hasPreviousPosition <{pos1}> .")

        insert_query = f"""PREFIX trajecto: <http://w3id.org/trajecto#>

INSERT DATA {{
{chr(10).join(insert_statements)}
}}"""

        success = execute_sparql_update(update_endpoint, insert_query)
        if success:
            total_links_inserted += len(batch) * 2  # 2 triples per pair
            print(f"   Link batch {batch_start // batch_size + 1}: inserted {len(batch) * 2} triples")
        else:
            print(f"   Link batch {batch_start // batch_size + 1}: FAILED")

    print(f"\n✓ Total indices added: {total_indices_inserted}")
    print(f"✓ Total links created: {total_links_inserted}")

    return {
        "positions": len(df),
        "trajectories": len(trajectory_positions),
        "pairs": len(pairs),
        "indices_added": total_indices_inserted,
        "links_created": total_links_inserted
    }


print("Position sequence linking function defined.")

In [ ]:
# Execute the Python-based position sequence linking
# Uncomment below to run (comment out position-secuences.sparql in query_names list above first)

stats = link_position_sequences(query_endpoint, update_endpoint, batch_size=500)
print(f"\nStatistics: {stats}")

In [ ]:
# Execute queries
results = {}

for query_name in query_names:
    print(f"\n{'='*50}")
    print(f"Executing: {query_name}")
    print(f"{'='*50}")

    query = load_query(query_name)

    json_result = execute_sparql_update(update_endpoint, query)
    if json_result:
        print("✓ Query executed)")

In [ ]:
# Results summary
print(f"\n\n{'='*50}")
print("EXECUTION SUMMARY")
print(f"{'='*50}")

for query_name, result in results.items():
    if result is not None and not result.empty:
        print(f"✓ {query_name}: {len(result)} rows")
    else:
        print(f"✗ {query_name}: Failed or no results")

print("\nResults available in 'results' dictionary")
print(f"Access with: results['{query_names[0]}']")

In [ ]:
# Query to retrieve linked POIs with external data

sample_query = """
PREFIX poi: <http://w3id.org/trajecto/poi#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX trajecto: <http://w3id.org/trajecto#>

SELECT DISTINCT ?poi ?poiName ?externalEntity ?externalLabel ?externalDescription
WHERE {
    ?poi a poi:PointOfInterest .

    OPTIONAL { ?poi rdfs:label ?poiName }

    # Get external entity links
    OPTIONAL {
        ?poi owl:sameAs ?externalEntity .

        OPTIONAL { ?poi trajecto:externalLabel ?externalLabel }
        OPTIONAL { ?poi trajecto:externalDescription ?externalDescription }
    }

    # Only show linked POIs
    FILTER(BOUND(?externalEntity))
}
LIMIT 10
"""

print("\nSample Linked POIs:")
print("="*60)

results = execute_sparql_query(query_endpoint, sample_query)
df_linked = format_results(results)

if not df_linked.empty:
    for _idx, row in df_linked.iterrows():
        print(f"\nPOI: {row.get('poiName', 'N/A')}")
        print(f"  External Entity: {row.get('externalEntity', 'N/A')}")
        if row.get('externalLabel'):
            print(f"  Label: {row.get('externalLabel')}")
        if row.get('externalDescription'):
            desc = row.get('externalDescription')
            # Truncate long descriptions
            if len(desc) > 100:
                desc = desc[:100] + "..."
            print(f"  Description: {desc}")
else:
    print("No linked POIs found yet. The linking queries may need more time to complete.")

print(f"\nTotal linked POIs shown: {len(df_linked)}")

### Entity Linking Statistics

View how many POIs were successfully linked to external knowledge bases.

In [ ]:
# Wikidata Linking via Geo-coordinates and Name Matching
# Fetches POIs from interactions and creates owl:sameAs links to Wikidata

import hashlib
import json
import logging
import time
from pathlib import Path
from typing import Optional

import pandas as pd
from fuzzywuzzy import fuzz
from helpers.fuseki_config import query_endpoint, update_endpoint
from helpers.fuseki_queries import execute_sparql_query, execute_sparql_update, format_results
from rdflib import Graph, URIRef
from rdflib.namespace import OWL
from SPARQLWrapper import JSON, SPARQLWrapper

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
SEARCH_RADIUS_KM = 0.15  # ~150 meters
MIN_SIMILARITY_THRESHOLD = 80
REQUEST_DELAY = 1  # seconds between requests (be nice to Wikidata)

# Cache configuration
CACHE_DIR = Path("./wikidata_cache")
CACHE_DIR.mkdir(exist_ok=True)
COORDINATE_PRECISION = 5  # Decimal places for coordinate rounding (5 = ~1.1m precision)

# In-memory cache for current session
_wikidata_cache = {}


def _get_cache_key(lat: float, lon: float, radius_km: float) -> str:
    """Generate a cache key based on rounded coordinates and radius."""
    # Round coordinates to avoid cache misses for very close locations
    lat_rounded = round(lat, COORDINATE_PRECISION)
    lon_rounded = round(lon, COORDINATE_PRECISION)
    key_str = f"{lat_rounded}_{lon_rounded}_{radius_km}"
    return hashlib.md5(key_str.encode()).hexdigest()


def _get_cache_file(cache_key: str) -> Path:
    """Get the cache file path for a given key."""
    return CACHE_DIR / f"{cache_key}.json"


def load_from_cache(lat: float, lon: float, radius_km: float) -> Optional[list[dict]]:
    """Load Wikidata results from cache if available."""
    cache_key = _get_cache_key(lat, lon, radius_km)

    # Check in-memory cache first
    if cache_key in _wikidata_cache:
        logger.debug(f"  Cache hit (memory): {cache_key[:8]}...")
        return _wikidata_cache[cache_key]

    # Check file cache
    cache_file = _get_cache_file(cache_key)
    if cache_file.exists():
        try:
            with open(cache_file, encoding='utf-8') as f:
                data = json.load(f)
                _wikidata_cache[cache_key] = data  # Store in memory too
                logger.debug(f"  Cache hit (file): {cache_key[:8]}...")
                return data
        except (OSError, json.JSONDecodeError) as e:
            logger.warning(f"  Cache read error: {e}")

    return None


def save_to_cache(lat: float, lon: float, radius_km: float, results: list[dict]):
    """Save Wikidata results to cache."""
    cache_key = _get_cache_key(lat, lon, radius_km)

    # Save to memory
    _wikidata_cache[cache_key] = results

    # Save to file
    cache_file = _get_cache_file(cache_key)
    try:
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False)
        logger.debug(f"  Cached: {cache_key[:8]}...")
    except OSError as e:
        logger.warning(f"  Cache write error: {e}")


def clear_cache():
    """Clear all cached Wikidata results."""
    global _wikidata_cache
    _wikidata_cache = {}
    for cache_file in CACHE_DIR.glob("*.json"):
        cache_file.unlink()
    logger.info("Wikidata cache cleared")


def get_cache_stats() -> dict:
    """Get statistics about the cache."""
    file_count = len(list(CACHE_DIR.glob("*.json")))
    memory_count = len(_wikidata_cache)
    return {
        "files_cached": file_count,
        "memory_entries": memory_count,
        "cache_dir": str(CACHE_DIR)
    }


print(f"Cache directory: {CACHE_DIR}")
print(f"Coordinate precision: {COORDINATE_PRECISION} decimal places")
stats = get_cache_stats()
print(f"Cached entries: {stats['files_cached']} files, {stats['memory_entries']} in memory")

In [ ]:
# Fetch POIs from interactions in the graph


def fetch_pois_from_interactions() -> pd.DataFrame:
    """Fetch POIs that are linked to interactions via poi:interactsWith."""

    query = """
    PREFIX trajecto: <http://w3id.org/trajecto#>
    PREFIX poi: <http://w3id.org/trajecto/poi#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX datacron: <http://www.datacron-project.eu/datAcron#>


    SELECT DISTINCT ?poi ?label ?altLabel ?lat ?lon
    WHERE {
        ?interaction poi:interactsWith ?poi .
        ?poi a trajecto:PointOfInterest ;
            datacron:hasLatitude ?lat;
            datacron:hasLongitude ?lon.

        OPTIONAL { ?poi rdfs:label ?label }
        OPTIONAL { ?poi skos:altLabel ?altLabel }
    }
    """

    results = execute_sparql_query(query_endpoint, query)
    df = format_results(results)

    if not df.empty:
        # Rename columns to match expected format
        df = df.rename(columns={
            'poi': 'iri',
            'altLabel': 'alt_label'
        })

        # Convert lat/lon to float
        if 'lat' in df.columns:
            df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
        if 'lon' in df.columns:
            df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

    return df


# Test fetching POIs
pois_df = fetch_pois_from_interactions()
print(f"Found {len(pois_df)} POIs from interactions")
if not pois_df.empty:
    print(pois_df.head())

In [ ]:
# Wikidata matching functions


def query_wikidata_nearby(lat: float, lon: float, radius_km: float = SEARCH_RADIUS_KM, use_cache: bool = True) -> list[dict]:
    """
    Query Wikidata for entities near a coordinate.
    Results are cached to prevent redundant API calls.
    """

    # Check cache first
    if use_cache:
        cached_results = load_from_cache(lat, lon, radius_km)
        if cached_results is not None:
            logger.info(f"  [CACHE HIT] Using cached results for ({lat:.5f}, {lon:.5f})")
            return cached_results

    query = f"""
    SELECT ?entity ?label ?labelEn ?coord ?distance
    WHERE {{
      SERVICE wikibase:around {{
        ?entity wdt:P625 ?coord .
        bd:serviceParam wikibase:center "Point({lon} {lat})"^^geo:wktLiteral .
        bd:serviceParam wikibase:radius "{radius_km}" .
      }}
        # Native label (P1705)
      OPTIONAL {{ ?entity wdt:P1705 ?label . }}
      OPTIONAL {{ ?entity rdfs:label ?labelEn . FILTER(LANG(?labelEn) = "en") }}

      BIND(geof:distance("Point({lon} {lat})"^^geo:wktLiteral, ?coord) AS ?distance)
    }}
    ORDER BY ?distance
    LIMIT 50
    """

    sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.addCustomHttpHeader("User-Agent", "POI-Matcher/1.0 (Academic Research)")

    try:
        results = sparql.query().convert()
        bindings = results["results"]["bindings"]

        # Save to cache
        if use_cache:
            save_to_cache(lat, lon, radius_km, bindings)
            logger.info(f"  [CACHED] Saved {len(bindings)} results for ({lat:.5f}, {lon:.5f})")

        return bindings
    except Exception as e:
        logger.error(f"SPARQL query failed: {e}")
        return []


def calculate_similarity(target: str, candidate: str) -> float:
    """Calculate string similarity using multiple methods."""
    if not target or not candidate:
        return 0.0

    # Normalize strings
    target = target.strip().lower()
    candidate = candidate.strip().lower()

    # Check containment first (high confidence)
    if target in candidate or candidate in target:
        return 95.0

    # Fuzzy matching
    ratio = fuzz.ratio(target, candidate)
    token_ratio = fuzz.token_set_ratio(target, candidate)
    partial_ratio = fuzz.partial_ratio(target, candidate)

    # Weighted average favoring token_set_ratio (handles word reordering)
    return max(ratio, token_ratio * 0.95, partial_ratio * 0.9)


def find_best_match(
    native_label: str,
    english_label: Optional[str],
    wikidata_results: list[dict],
    threshold: float = MIN_SIMILARITY_THRESHOLD
) -> Optional[dict]:
    """Find the best matching Wikidata entity for a POI."""

    best_match = None
    best_score = 0.0

    for result in wikidata_results:
        entity_uri = result["entity"]["value"]
        wd_label_native = result.get("label", {}).get("value", "")
        wd_label_en = result.get("labelEn", {}).get("value", "")
        distance = float(result.get("distance", {}).get("value", 999))

        # Calculate similarity scores
        score_native_native = calculate_similarity(native_label, wd_label_native)
        score_native_en = calculate_similarity(native_label, wd_label_en)
        score_en_native = calculate_similarity(english_label, wd_label_native) if english_label else 0
        score_en_en = calculate_similarity(english_label, wd_label_en) if english_label else 0

        # Take the best score across all label combinations
        max_score = max(score_native_native, score_native_en, score_en_native, score_en_en)

        # Slight penalty for distance (closer is better for tie-breaking)
        adjusted_score = max_score - (distance / 100)

        if adjusted_score > best_score and max_score >= threshold:
            best_score = adjusted_score
            best_match = {
                "wikidata_uri": entity_uri,
                "wikidata_label_native": wd_label_native,
                "wikidata_label_en": wd_label_en,
                "similarity_score": max_score,
                "distance_m": distance
            }

    return best_match


print("Wikidata matching functions defined (with caching).")

In [ ]:
# Main matching function - returns only owl:sameAs triples


def match_pois_to_wikidata(df: pd.DataFrame, use_cache: bool = True) -> tuple[pd.DataFrame, Graph]:
    """
    Match POIs from DataFrame to Wikidata entities.
    Returns only owl:sameAs relations (no other data).
    Uses caching to prevent redundant Wikidata API calls.

    Expected DataFrame columns:
    - iri: POI IRI/URI
    - label: Native language label
    - alt_label: English label (optional, can be NaN)
    - lat: Latitude
    - lon: Longitude
    """

    # Initialize RDF graph for owl:sameAs triples ONLY
    g = Graph()
    g.bind("owl", OWL)

    # Results tracking
    matches = []
    cache_hits = 0
    api_calls = 0

    for idx, row in df.iterrows():
        poi_iri = row["iri"]
        native_label = row.get("label", "")
        english_label = row.get("alt_label") if pd.notna(row.get("alt_label")) else None
        lat = row.get("lat")
        lon = row.get("lon")

        # Skip if missing coordinates or label
        if pd.isna(lat) or pd.isna(lon) or not native_label:
            logger.warning(f"Skipping POI {poi_iri}: missing coordinates or label")
            continue

        logger.info(f"Processing [{idx+1}/{len(df)}]: {native_label}")

        # Check if this will be a cache hit (for statistics)
        is_cache_hit = use_cache and load_from_cache(lat, lon, SEARCH_RADIUS_KM) is not None

        # Query Wikidata for nearby entities (uses cache internally)
        wikidata_results = query_wikidata_nearby(lat, lon, use_cache=use_cache)

        if is_cache_hit:
            cache_hits += 1
        else:
            api_calls += 1

        if not wikidata_results:
            logger.warning("  No Wikidata entities found nearby")
            matches.append({
                "poi_iri": poi_iri,
                "poi_label": native_label,
                "matched": False,
                "wikidata_uri": None,
                "wikidata_label": None,
                "similarity_score": None,
                "distance_m": None
            })
            # Only delay for actual API calls, not cache hits
            if not is_cache_hit:
                time.sleep(REQUEST_DELAY)
            continue

        # Find best match
        best_match = find_best_match(native_label, english_label, wikidata_results)

        if best_match:
            logger.info(f"  ✓ Match: {best_match['wikidata_label_native'] or best_match['wikidata_label_en']} "
                       f"(score: {best_match['similarity_score']:.1f}%, dist: {best_match['distance_m']:.1f}m)")

            # Add ONLY owl:sameAs triple - no other data
            g.add((
                URIRef(poi_iri),
                OWL.sameAs,
                URIRef(best_match["wikidata_uri"])
            ))

            matches.append({
                "poi_iri": poi_iri,
                "poi_label": native_label,
                "matched": True,
                "wikidata_uri": best_match["wikidata_uri"],
                "wikidata_label": best_match["wikidata_label_native"] or best_match["wikidata_label_en"],
                "similarity_score": best_match["similarity_score"],
                "distance_m": best_match["distance_m"]
            })
        else:
            logger.info("  ✗ No match above threshold")
            matches.append({
                "poi_iri": poi_iri,
                "poi_label": native_label,
                "matched": False,
                "wikidata_uri": None,
                "wikidata_label": None,
                "similarity_score": None,
                "distance_m": None
            })

        # Only delay for actual API calls, not cache hits
        if not is_cache_hit:
            time.sleep(REQUEST_DELAY)

    results_df = pd.DataFrame(matches)

    # Log cache statistics
    logger.info(f"\nCache statistics: {cache_hits} cache hits, {api_calls} API calls")
    if cache_hits + api_calls > 0:
        logger.info(f"Cache hit rate: {cache_hits / (cache_hits + api_calls) * 100:.1f}%")

    return results_df, g


print("Main matching function defined (with caching support).")

In [ ]:
# Function to push owl:sameAs triples to Fuseki


def push_sameas_to_fuseki(graph: Graph, update_endpoint: str) -> bool:
    """
    Push only owl:sameAs triples from the graph to Fuseki.
    This function ensures only sameAs relations are inserted as proper URIs, not literals.
    """

    # Extract only owl:sameAs triples
    sameas_triples = list(graph.triples((None, OWL.sameAs, None)))

    if not sameas_triples:
        logger.warning("No owl:sameAs triples to push")
        return False

    # Build INSERT DATA query with only sameAs triples
    # Ensure both subject and object are formatted as URIs (not literals)
    insert_statements = []
    for subj, _pred, obj in sameas_triples:
        # Convert to string and wrap in angle brackets for proper URI syntax
        subj_uri = str(subj)
        obj_uri = str(obj)
        insert_statements.append(f"  <{subj_uri}> owl:sameAs <{obj_uri}> .")

    insert_query = f"""PREFIX owl: <http://www.w3.org/2002/07/owl#>

INSERT DATA {{
{chr(10).join(insert_statements)}
}}"""

    logger.info(f"Pushing {len(sameas_triples)} owl:sameAs triples to Fuseki...")
    logger.debug(f"Sample triple: {insert_statements[0] if insert_statements else 'none'}")

    success = execute_sparql_update(update_endpoint, insert_query)

    if success:
        logger.info(f"✓ Successfully pushed {len(sameas_triples)} owl:sameAs relations")
    else:
        logger.error("✗ Failed to push owl:sameAs relations to Fuseki")

    return success


print("Push function defined.")

In [ ]:
# Execute the full Wikidata linking pipeline

print("="*60)
print("WIKIDATA ENTITY LINKING")
print("="*60)

# Show cache status
cache_stats = get_cache_stats()
print(f"\nCache status: {cache_stats['files_cached']} cached locations")

# Step 1: Fetch POIs from interactions
print("\n1. Fetching POIs from interactions...")
pois_df = fetch_pois_from_interactions()
print(f"   Found {len(pois_df)} POIs")

if pois_df.empty:
    print("   No POIs found. Ensure interactions exist with poi:interactsWith relations.")
else:
    # Step 2: Match POIs to Wikidata
    print("\n2. Matching POIs to Wikidata...")
    print(f"   Search radius: {SEARCH_RADIUS_KM * 1000}m")
    print(f"   Similarity threshold: {MIN_SIMILARITY_THRESHOLD}%")
    print(f"   Request delay: {REQUEST_DELAY}s (skipped for cache hits)")
    print("   Caching: ENABLED")

    results_df, sameas_graph = match_pois_to_wikidata(pois_df, use_cache=True)

    # Step 3: Display results summary
    print("\n" + "="*60)
    print("MATCHING RESULTS")
    print("="*60)

    matched = results_df[results_df['matched']]
    unmatched = results_df[~results_df['matched']]

    print(f"Total POIs processed: {len(results_df)}")
    print(f"Successfully matched: {len(matched)} ({len(matched)/len(results_df)*100:.1f}%)")
    print(f"No match found: {len(unmatched)}")

    # Updated cache stats
    cache_stats = get_cache_stats()
    print(f"\nCache now contains: {cache_stats['files_cached']} locations")

    if not matched.empty:
        print(f"\nAverage similarity score: {matched['similarity_score'].mean():.1f}%")
        print(f"Average distance: {matched['distance_m'].mean():.1f}m")

        print("\nMatched POIs:")
        for _, row in matched.iterrows():
            print(f"  • {row['poi_label']} → {row['wikidata_label']} ({row['similarity_score']:.0f}%)")

    # Step 4: Push to Fuseki
    print("\n" + "="*60)
    print("PUSHING TO FUSEKI")
    print("="*60)

    if len(sameas_graph) > 0:
        success = push_sameas_to_fuseki(sameas_graph, update_endpoint)
        if success:
            print(f"\n✓ {len(list(sameas_graph.triples((None, OWL.sameAs, None))))} owl:sameAs relations added to graph")
    else:
        print("No owl:sameAs relations to push.")

In [ ]:
# Validate KG state against stage-03-enrichment SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-03-enrichment')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-03-enrichment', halt_on_violations=True)